In [110]:
from transformers import pipeline
import re
from rapidfuzz import fuzz

In [111]:
def normalize_text(text: str) -> str:
    text = text.lower()

    replacements = {
        '3': 'e', '4': 'a', '5': 's',
        '0': 'o', '@': 'a', '$': 's'
    }
    for k, v in replacements.items():
        text = text.replace(k, v)

    text = text.replace('1', 'l')

    text = re.sub(r'[^a-z0-9\s]', '', text)

    text = re.sub(
        r'(\b[a-z]\s+){2,}[a-z]\b',
        lambda m: m.group(0).replace(" ", ""),
        text
    )

    text = re.sub(r'\s+', ' ', text).strip()

    return text


def fuzzy_check(text):
    # targets = ["kill", "suicide", "rape", "nude"]
    targets = []

    for word in targets:
        score = max(
            fuzz.partial_ratio(text, word),
            fuzz.ratio(text, word)
        )

        if score > 80:
            return True, word

    return False, None

In [112]:
model_3 = pipeline(
    "text-classification",
    model="cointegrated/rubert-tiny-toxicity",
    return_all_scores=True
)

def extract_scores(output):
    """
    Handles both:
    - single label output
    - multi-label output
    """

    # Case 1: list of dicts (correct multi-label)
    if isinstance(output, list) and isinstance(output[0], dict):
        return {r["label"].lower(): r["score"] for r in output}

    # Case 2: nested list
    if isinstance(output, list) and isinstance(output[0], list):
        return {r["label"].lower(): r["score"] for r in output[0]}

    # Case 3: single dict
    if isinstance(output, dict):
        return {output["label"].lower(): output["score"]}

    return {}

# def predict_model_3(text):
#     output = model_3(text, top_k=None)

#     if isinstance(output[0], list):
#         output = output[0]

#     scores = {o["label"]: o["score"] for o in output}

#     return {
#         "toxicity": 1 - scores.get("non-toxic", 1.0),
#         "dangerous": scores.get("dangerous", 0.0),
#         "threat": scores.get("threat", 0.0),
#         "insult": scores.get("insult", 0.0),
#         "obscenity": scores.get("obscenity", 0.0)
#     }

def predict_model_3(text):
    output = model_3(text, top_k=None)

    if isinstance(output[0], list):
        output = output[0]

    return {o["label"]: o["score"] for o in output}

Loading weights: 100%|██████████| 57/57 [00:00<00:00, 5575.58it/s]
BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny-toxicity
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [113]:
def decide(scores):

    # 🚨 HARD BLOCK (high severity)
    if scores["threat"] > 0.2:
        return "BLOCKED", "threat"

    if scores["dangerous"] > 0.6:
        return "BLOCKED", "dangerous"

    if scores["obscenity"] > 0.7:
        return "BLOCKED", "obscenity"

    # ⚠️ FLAG (moderate issues)
    if scores["insult"] > 0.2:
        return "FLAGGED", "insult"

    if scores["dangerous"] > 0.3:
        return "FLAGGED", "dangerous"

    # ✅ SAFE
    return "APPROVED", None

In [114]:
def moderate_with_model(text, predict_fn):
    processed = normalize_text(text)

    # Fuzzy check first
    fuzzy_hit, word = fuzzy_check(processed)
    if fuzzy_hit:
        return {
            "text": text,
            "processed": processed,
            "status": "BLOCKED",
            "reason": "fuzzy_match",
            "word": word
        }

    # Model prediction
    score = predict_fn(processed)

    return {
        "text": text,
        "processed": processed,
        "toxicity": score,
        "status": decide(score)
    }

In [115]:
tests = [
    "Hello, how are you?",
    "I hate you",
    "You are stupid",
    "Send me nudes",
    "I will kill you",
    "k i l l",
    "k!ll",
    "ki11",
    "Let's build something amazing",
    "You are annoying",
    "Go to hell",
    "kill time"
]
print("Model 3 output :-")
for t in tests:
    print("TEXT:", t)
    print("Model 3:", moderate_with_model(t, predict_model_3))

    print("-" * 80)

Model 3 output :-
TEXT: Hello, how are you?
Model 3: {'text': 'Hello, how are you?', 'processed': 'hello how are you', 'toxicity': {'non-toxic': 0.9801693558692932, 'dangerous': 0.23001715540885925, 'insult': 0.017758501693606377, 'obscenity': 0.00023239130678121, 'threat': 0.00013006902008783072}, 'status': ('APPROVED', None)}
--------------------------------------------------------------------------------
TEXT: I hate you
Model 3: {'text': 'I hate you', 'processed': 'i hate you', 'toxicity': {'non-toxic': 0.9922935962677002, 'dangerous': 0.2540779709815979, 'insult': 0.002723491517826915, 'obscenity': 0.0018716950435191393, 'threat': 0.0005369582213461399}, 'status': ('APPROVED', None)}
--------------------------------------------------------------------------------
TEXT: You are stupid
Model 3: {'text': 'You are stupid', 'processed': 'you are stupid', 'toxicity': {'non-toxic': 0.9961355924606323, 'dangerous': 0.1879570484161377, 'insult': 0.004222830291837454, 'threat': 0.0001267580

In [125]:

print("\nMODEL 3 OUTPUT:")
out = model_3("I will kill you", top_k=None)
print(out)


MODEL 3 OUTPUT:
[{'label': 'threat', 'score': 0.9482415914535522}, {'label': 'dangerous', 'score': 0.5018850564956665}, {'label': 'obscenity', 'score': 0.07216811180114746}, {'label': 'non-toxic', 'score': 0.0302432831376791}, {'label': 'insult', 'score': 0.018314924091100693}]
